In [1]:
from pathlib import Path
import shutil, os
# Reuse the pinned source and cached downloads on the attached Volume.
shutil.copytree("/mnt/m3diff/baseline_reproduction_v1/repo", "/root/M3Diff", ignore=shutil.ignore_patterns(".git", "__pycache__"))


'/root/M3Diff'

In [2]:
import os, sys, json, shutil, subprocess, glob, time
VOL   = "/mnt/m3diff"
REPO  = "/root/M3Diff"
VENV  = "/root/m3diff-venv"
PY    = f"{VENV}/bin/python"
CKPT  = "/root/ckpt/M3Diff"
SYN   = f"{VOL}/synth"            # Notebook 1
FT    = f"{VOL}/ft"               # Notebook này
CODE  = f"{FT}/code"
OUT   = f"{FT}/m3diff-ft-lora"    # tên thư mục phải chứa 'm3diff' và 'lora' để repo nạp đúng kiểu LoRA
EVAL  = f"{FT}/eval"
OBJ_ALL   = f"{FT}/objects_all"
TRAIN_SYN = f"{FT}/train_synth"
CAMTEST   = f"{FT}/camtest"
MIX       = f"{FT}/train_mix.json"
QUANT, DTYPE = "none", "bfloat16"

# Siêu tham số
SCALE    = 1.0        # nhân số cặp train tổng hợp (1.0 ≈ 2.700 cặp)
N_REPLAY = 2700       # số câu OmniDiff train gốc trộn vào
LORA_R, LORA_ALPHA = 64, 128
LR, PROJ_LR, EPOCHS = 4e-5, 1e-5, 1
HF_TOKEN = os.environ.get("HF_TOKEN", "")
os.environ["MPLBACKEND"] = "Agg"
os.environ["TQDM_MININTERVAL"] = "20"
for d in [CODE, EVAL, f"{FT}/results"]:
    os.makedirs(d, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "KHÔNG CÓ GPU")
mem = int(gpu.split(",")[-1]) if gpu else 0
BATCH = 4 if mem >= 70000 else (2 if mem >= 39000 else 1)
ACCUM = 16 // BATCH
print(f"batch/GPU = {BATCH}, tích luỹ gradient = {ACCUM} (batch hiệu dụng 16)")
if mem < 39000:
    print("!! GPU dưới 40GB: train có thể hết bộ nhớ. Nên chọn A100.")

need = {"code Notebook 1": f"{SYN}/code/make_synth.py", "bộ test tổng hợp": f"{SYN}/data/manifest.jsonl",
        "câu model gốc trên bộ tổng hợp": f"{SYN}/data/answers.jsonl", "điểm model gốc (per_pair)": f"{SYN}/results/per_pair.csv",
        "vật thể test": f"{SYN}/objects/objects.csv", "kết quả gốc Real": f"{VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl"}
for k, v in need.items():
    print(("OK    " if os.path.exists(v) else "THIẾU ") + f"{k}: {v}")
print(("OK    " if os.path.exists(f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl") else "(không có, bỏ qua Render) ")
      + "kết quả gốc Render")

GPU: NVIDIA A10, 23028
batch/GPU = 1, tích luỹ gradient = 16 (batch hiệu dụng 16)
!! GPU dưới 40GB: train có thể hết bộ nhớ. Nên chọn A100.
OK    code Notebook 1: /mnt/m3diff/synth/code/make_synth.py
OK    bộ test tổng hợp: /mnt/m3diff/synth/data/manifest.jsonl
OK    câu model gốc trên bộ tổng hợp: /mnt/m3diff/synth/data/answers.jsonl
OK    điểm model gốc (per_pair): /mnt/m3diff/synth/results/per_pair.csv
OK    vật thể test: /mnt/m3diff/synth/objects/objects.csv
OK    kết quả gốc Real: /mnt/m3diff/results/OmniDiff_Real/OmniDiff_Real.jsonl
OK    kết quả gốc Render


In [3]:
%pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"
!{UV} pip install -q --python {PY} peft==0.4.0 "numpy<2" "opencv-python-headless<4.12" pandas scipy matplotlib pycocotools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


/root/M3Diff


torch 2.2.2+cu121 | transformers 4.40.0 | numpy 1.26.4


CUDA: True | NVIDIA A10 | 22.1 GB


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/__init__.py", line 22, in <module>
    from .auto import (
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/auto.py", line 30, in <module>
    from .mapping import MODEL_TYPE_TO_PEFT_MODEL_MAPPING
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/mapping.py", line 20, in <module>
    from .peft_model import (
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/peft_model.py", line 39, in <module>
    from .tuners import (
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/tuners/__init__.py", line 21, in <module>
    from .lora import LoraConfig, LoraModel
  File "/root/m3diff-venv/lib/python3.10/site-packages/peft/tuners/lora.py", line 42, in <module>
    import bitsandbytes as bnb
  File "/root/m3diff-venv/lib/python3.10/site-packages/bitsandbytes/__init__.py", line 15, in <module>
    from .nn import modules
  Fil

In [4]:
!{UV} pip install -q --python {PY} setuptools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

peft 0.4.0 | numpy 1.26.4 | cv2 4.11.0


In [5]:
import hashlib
os.makedirs('/root/ckpt', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/M3Diff', CKPT)
os.makedirs(f'{REPO}/datasets', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/OmniDiff', f'{REPO}/datasets/OmniDiff')
assert os.path.isfile(f'{CKPT}/generation_config.json')
assert os.path.isfile(f'{CAMTEST}/questions.json')
print('Bước 3: dùng checkpoint và ảnh đã cache trên Volume; không tải lại.')


Bước 3: dùng checkpoint và ảnh đã cache trên Volume; không tải lại.


In [6]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

Patch OK: True True


In [7]:
from IPython.display import Image, display
import pandas as pd
os.makedirs(f"{REPO}/datasets", exist_ok=True)
link = f"{REPO}/datasets/ft_synth"          # ảnh train tổng hợp nằm trên Volume, nối vào thư mục datasets
if os.path.isdir(TRAIN_SYN) and not os.path.islink(link):
    os.symlink(TRAIN_SYN, link)

def run_eval(qfile, answers, image_folder, lora):
    mp = f"--model-path {OUT} --model-base {CKPT}" if lora else f"--model-path {CKPT}"
    cmd = (f"cd {REPO} && PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py "
           f"{mp} --question-file {qfile} --answers-file {answers} --image-folder {image_folder} "
           f"--num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention")
    n_total = len(json.load(open(qfile)))
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    if done >= n_total:
        print(f"Đã đủ {done}/{n_total}: {answers}"); return
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True).returncode
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    print(f"-> {done}/{n_total} câu, {(time.time()-t0)/60:.1f} phút, mã thoát {rc}")

MDP_ARGS = '{"num_refinement_layers":2,"feature_dim":1152,"ffn_hidden_dim":4608,"attention_downsample_rate":1}'

def train_args(out_dir, extra=()):
    return [PY, f"{CODE}/ft_train.py",
        "--model_name_or_path", CKPT, "--version", "qwen_2", "--output_dir", out_dir,
        "--data_path", MIX, "--image_folder", f"{REPO}/datasets", "--image_aspect_ratio", "pad",
        "--vision_tower", "google/siglip-so400m-patch14-384", "--mm_vision_select_layer", "[-11,-8,-5,-2]",
        "--unfreeze_mm_vision_tower", "False", "--mm_projector_type", "mlp2x_gelu", "--mm_projector_lr", str(PROJ_LR),
        "--mm_patch_merge_type", "flat", "--mm_resampler_type", "mdp", "--mdp_args", MDP_ARGS,
        "--tune_mm_vision_resampler", "True", "--mm_use_im_start_end", "False", "--mm_use_im_patch_token", "False",
        "--lora_enable", "True", "--lora_r", str(LORA_R), "--lora_alpha", str(LORA_ALPHA),
        "--num_train_epochs", str(EPOCHS), "--per_device_train_batch_size", str(BATCH),
        "--gradient_accumulation_steps", str(ACCUM), "--learning_rate", str(LR), "--weight_decay", "0.",
        "--warmup_ratio", "0.03", "--lr_scheduler_type", "cosine", "--logging_steps", "5",
        "--model_max_length", "4096", "--gradient_checkpointing", "True", "--group_by_modality_length", "True",
        "--dataloader_num_workers", "4", "--lazy_preprocess", "True", "--bf16", "True", "--tf32", "True",
        "--report_to", "none", *extra]

def run(args):
    from collections import deque
    env = dict(os.environ, ATTN="sdpa", PYTHONWARNINGS="ignore", TOKENIZERS_PARALLELISM="false")
    t0, tail = time.time(), deque(maxlen=40)
    p = subprocess.Popen(args, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("'loss'", "train_runtime", "out of memory", "resume", "it/s", "s/it", "Adding LoRA")
    for line in p.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in keep):
            print(line.rstrip()[-300:], flush=True)
    p.wait()
    if p.returncode != 0:
        print("\n----- LỖI: 40 dòng cuối -----\n" + "\n".join(tail))
    print(f"[xong] mã thoát {p.returncode}, {(time.time()-t0)/60:.1f} phút")
    return p.returncode
print("OK")

OK


In [8]:
# Mandatory stop instead of allowing Step 11 to start training.
assert os.path.isfile(f'{OUT}/adapter_model.bin') or os.path.isfile(f'{OUT}/adapter_model.safetensors'), 'STOP: chưa có checkpoint đã train'
state=json.load(open(f'{OUT}/trainer_state.json'))
assert state['global_step']==state['max_steps']==338, 'STOP: training not complete'
if not os.path.islink(f"{REPO}/datasets/ft_synth"):
    os.symlink(TRAIN_SYN, f"{REPO}/datasets/ft_synth")
if os.path.exists(f"{OUT}/adapter_model.bin") or os.path.exists(f"{OUT}/adapter_model.safetensors"):
    print("Đã train xong:", OUT)
else:
    rc = run(train_args(OUT, ["--save_strategy", "steps", "--save_steps", "50", "--save_total_limit", "1"]))

# Đảm bảo khi nạp lại, vision tower lấy đúng trọng số đã fine-tune của M3Diff gốc (nạp sớm từ checkpoint)
cfg_p = f"{OUT}/config.json"
if os.path.exists(cfg_p):
    cfg = json.load(open(cfg_p)); cfg["unfreeze_mm_vision_tower"] = True
    json.dump(cfg, open(cfg_p, "w"), indent=2)
    print("Đã lưu model:", sorted(os.listdir(OUT)))

Đã train xong: /mnt/m3diff/ft/m3diff-ft-lora


Đã lưu model: ['README.md', 'adapter_config.json', 'adapter_model.bin', 'checkpoint-300', 'config.json', 'non_lora_trainables.bin', 'trainer_state.json']


In [9]:
# Fail fast when an eval subprocess fails or output IDs are incomplete.
_run_eval=run_eval
def run_eval(qfile, answers, image_folder, lora):
    start=time.time()
    _run_eval(qfile, answers, image_folder, lora)
    q=json.load(open(qfile)); rows=[json.loads(l) for l in open(answers) if l.strip()]
    assert len(rows)==len(q) and {str(r['id']) for r in rows}=={str(r['id']) for r in q}, 'Incomplete evaluation: STOP'
    assert all(r['caption'].strip() for r in rows)
    timings.setdefault('evals',[]).append({'questions':qfile,'answers':answers,'count':len(rows),'wall_seconds':time.time()-start})
    json.dump(timings,open(f'{RUN}/timings.json','w'),indent=2)
RUN='/mnt/m3diff/r64_camera_eval_20261102'
os.makedirs(RUN,exist_ok=True)
timings={'gpu':gpu,'source':'m3diff-nb-finetune.ipynb','evals':[]}


In [10]:
san = f"{EVAL}/sanity_ft.jsonl"
if os.path.exists(san):
    os.remove(san)
run_eval(f"{SYN}/sanity/questions.json", san, f"{SYN}/sanity", lora=True)
base_san = {json.loads(l)["id"]: json.loads(l)["caption"] for l in open(f"{SYN}/sanity/answers.jsonl")} if os.path.exists(f"{SYN}/sanity/answers.jsonl") else {}
for l in open(san):
    r = json.loads(l)
    print(f"\n[{r['id']}]\n  gốc      : {base_san.get(r['id'], '—')}\n  fine-tune: {r['caption']}")

/mnt/m3diff/ft/m3diff-ft-lora
[patch] quant_mode=none


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


The checkpoint contains an unfrozen vision tower; loading it eagerly.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.21s/it]


Loading additional LLaVA weights...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
[patch] resume: 0 done, 8 left


  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:14<00:00,  1.81s/it]


-> 8/8 câu, 0.9 phút, mã thoát 0



[add_L2_00]
  gốc      : A new cluster of yellow-green fish, swimming in a school, has appeared in the lower right corner of the image.
  fine-tune: A yellow bowl has been added in the lower right part of the scene.

[remove_L2_00]
  gốc      : The cup of milk that was in the middle of the two tree stumps has disappeared. The number of peanuts on the tree stump on the right side has increased.
  fine-tune: The orange cup in the center of the scene is no longer present.

[color_L2_00]
  gốc      : The peach slices that were originally yellow have turned green.
  fine-tune: The color of the apple in the center of the scene has changed from orange to green.

[move_L2_00]
  gốc      : The red car in the middle of the road has turned to face the right side of the scene. A larger yellow jar filled with an orange substance has moved from the rooftop of the building on the right side of the scene to the center of the road, closer to the intersection.
  fine-tune: The yellow cup in the center 

In [11]:
run_eval(f'{CAMTEST}/questions.json',f'{EVAL}/camtest_ft.jsonl',CAMTEST,lora=True)
for file in ['questions.json','manifest.jsonl']:
    if os.path.isfile(f'{CAMTEST}/{file}'): shutil.copy2(f'{CAMTEST}/{file}',f'{RUN}/{file}')
for file in ['camtest_base.jsonl','camtest_ft.jsonl','sanity_ft.jsonl']:
    shutil.copy2(f'{EVAL}/{file}',f'{RUN}/{file}')
subprocess.run([PY,f'{CODE}/score_camera.py','--root',CAMTEST,'--base',f'{EVAL}/camtest_base.jsonl','--ft',f'{EVAL}/camtest_ft.jsonl','--synth-code',f'{SYN}/code','--out',f'{RUN}/camtest_compare.csv'],check=True)
json.dump({'complete':True,'camera_count':240,'sanity_count':8,'full_omnidiff_run':False},open(f'{RUN}/SUCCESS.json','w'),indent=2)
print('DONE: sanity8 + camera240 only. Release GPU now.')


/mnt/m3diff/ft/m3diff-ft-lora
[patch] quant_mode=none


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


The checkpoint contains an unfrozen vision tower; loading it eagerly.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.49s/it]


Loading additional LLaVA weights...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
[patch] resume: 0 done, 240 left


... (37 outputs left out)

 99%|█████████▉| 237/240 [06:37<00:05,  1.87s/it]

100%|██████████| 240/240 [06:43<00:00,  1.68s/it]


-> 240/240 câu, 6.9 phút, mã thoát 0


== Chỉ camera chuyển động (không có vật nào đổi) ==
Điều kiện   Mức   n |  nhắc camera  đúng hướng  bịa đổi vật | (gốc → fine-tune)
pan           1  15 |   0.0 →  80.0    0.0 →  33.3   73.3 →  13.3
pan           2  15 |   0.0 →  86.7    0.0 →  13.3   86.7 →   6.7
pan           3  15 |   0.0 →  93.3    0.0 →  40.0   86.7 →   6.7
rotate        1  15 |   0.0 →  73.3    0.0 →   6.7   46.7 →  26.7
rotate        2  15 |   0.0 →  86.7    0.0 →   0.0   73.3 →  20.0
rotate        3  15 |   0.0 →  93.3    0.0 →   0.0   73.3 →  40.0
zoom_in       1  15 |   0.0 →  66.7    0.0 →  40.0   86.7 →  33.3
zoom_in       2  15 |   0.0 →  60.0    0.0 →  33.3   73.3 →  46.7
zoom_in       3  15 |   0.0 →  80.0    0.0 →  53.3   93.3 →  26.7
zoom_out      1  15 |   0.0 →  93.3    0.0 →  20.0   53.3 →  20.0
zoom_out      2  15 |   0.0 →  86.7    0.0 →  33.3   53.3 →   6.7
zoom_out      3  15 |   0.0 →  93.3    0.0 →  26.7   93.3 →  40.0
Tổng 180 cặp — nhắc camera: 0.0% → 82.8%  (chỉ fine-tune đúng 149, chỉ gốc đ